In [2]:
%load_ext autoreload
%autoreload 2

import sys
import os
import getpass
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter Groq API Key: ")
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter Google API Key: ")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
%pip install pylate

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
dense_vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory=str(project_root / "data" / "chroma_naive_gemini"),
)

raw = dense_vectorstore.get(include=["documents"])
leaf_texts = raw["documents"]
print(f"{len(leaf_texts)} chunks pulled from the existing dense vectorstore")

1555 chunks pulled from the existing dense vectorstore


In [6]:
import json

raw = dense_vectorstore.get(include=["documents"])
with open(project_root / "data" / "leaf_texts.json", "w") as f:
    json.dump(raw["documents"], f)
print(f"Dumped {len(raw['documents'])} chunks")

Dumped 1555 chunks


In [17]:
# Execute PyLate index builder script
!~/my-jupyter-env/bin/python ~/Codes/rag-lab/rag_lab/retrieval/colbert_index_build.py

Failed to reload module 'pylate.models.Dense' from file '/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/pylate/models/Dense.py'
Traceback (most recent call last):
  File "/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/IPython/extensions/autoreload.py", line 326, in check
    superreload(m, reload, self.old_objects)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/IPython/extensions/autoreload.py", line 580, in superreload
    module = reload(module)
  File "/usr/lib/python3.14/importlib/__init__.py", line 129, in reload
    _bootstrap._exec(spec, module)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 869, in _exec
  File "<frozen importlib._bootstrap_external>", line 755, in exec_module
  File "<frozen importlib._bootstrap_external>", line 893, in get_code
  File "<frozen importlib._bootstrap_external>", line 823, in source_to_code
  File "<frozen importlib._bootstrap>",

Traceback (most recent call last):
  File "/home/mohnish/Codes/rag-lab/rag_lab/retrieval/colbert_index_build.py", line 22, in <module>
    from pylate import indexes, models
  File "/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/pylate/models/__init__.py", line 3, in <module>
    from .colbert import ColBERT
  File "/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/pylate/models/colbert.py", line 27, in <module>
    from .Dense import Dense
  File "/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/pylate/models/Dense.py", line 93
    config.get("activation_function", None) = import_from_string(
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
SyntaxError: cannot assign to function call here. Maybe you meant '==' instead of '='?


In [8]:
import json
from rag_lab.retrieval.colbert_backend import ColBERTVectorStoreShim

with open(project_root / "data" / "id_to_text.json") as f:
    id_to_text = json.load(f)

colbert_vectorstore = ColBERTVectorStoreShim(
    index_folder=str(project_root / "data" / "colbert_index"),
    id_to_text=id_to_text,
)

/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FileNotFoundError: [Errno 2] No such file or directory: '/home/mohnish/Codes/rag-lab/data/id_to_text.json'

In [ ]:
# Raw PyLate result inspection smoke test
raw = colbert_vectorstore.retriever.retrieve(
    queries_embeddings=colbert_vectorstore._embed_query("test query"),
    k=2,
)
print("Raw PyLate result — inspect field names:")
print(raw)

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

In [ ]:
# Test vectorstore shim retrieval interface
retriever = colbert_vectorstore.as_retriever(search_kwargs={"k": 2})
docs = retriever.invoke("test query")

for i, doc in enumerate(docs):
    print(f"--- Document {i+1} (Score: {doc.metadata.get('score')}) ---")
    print(doc.page_content[:200])
    print()